# ManūAI · 02 — Train

Runs `manuai.train` in this kernel and plots the run. Good for smoke tests and short experiments.
**Long runs** (hours) are better launched from `scripts/agent_runner.ipynb`, which runs detached under
`caffeinate` and survives closing VS Code. Harry's rule: big runs overnight.

To stop a run gracefully from anywhere: `touch data/logs/STOP` (keeps the best checkpoint).

In [ ]:
%load_ext autoreload
%autoreload 2
import json, os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Audio, display

from manuai import features as F
from manuai.config import REPO_ROOT, data_dir
from manuai.labels import class_names
os.chdir(REPO_ROOT)
CLASSES = class_names()
print(f"repo: {REPO_ROOT}\nclasses ({len(CLASSES)}): {', '.join(CLASSES)}")

## Configure

In [ ]:
MODEL     = "ast"                # "ast" (primary) or "vit" (baseline)
SMOKE     = True                 # 40-step check on the small cache
FEATURES  = "features_smoke" if SMOKE else "features"
RUN_NAME  = None                 # None → auto name, e.g. ast-20261012-1930-smoke
EXTRA     = []                   # e.g. ["--lr", "2e-4", "--epochs", "6", "--epoch-size", "19200"]

argv = ["--model", MODEL, "--features", FEATURES] + (["--smoke"] if SMOKE else []) + \
       (["--run-name", RUN_NAME] if RUN_NAME else []) + EXTRA
print("python -m manuai.train", " ".join(argv))

## Train
Guarded: flip `TRAIN = True` to run (uses the Mac's GPU).

In [ ]:
from manuai import train
TRAIN = False
run_dir = train.main(argv) if TRAIN else None

## Plot a run
Works for the run above or any existing one (e.g. the overnight `ast-v2-1`).

In [ ]:
runs = sorted((p.parent for p in (REPO_ROOT / "models/finetuned").glob("*/train_log.jsonl")),
              key=lambda p: p.stat().st_mtime)
RUN = run_dir or (runs[-1] if runs else None)
print("runs:", [r.name for r in runs]); print("plotting:", RUN)

In [ ]:
log = pd.DataFrame([json.loads(l) for l in open(RUN / "train_log.jsonl")])
train_rows = log[log["loss"].notna()] if "loss" in log else log.iloc[:0]
eval_rows = log[log["eval_macro_f1"].notna()] if "eval_macro_f1" in log else log.iloc[:0]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
if len(train_rows):
    ax[0].plot(train_rows.step, train_rows.loss)
ax[0].axhline(np.log(len(CLASSES)), ls=":", c="grey", label="ln K")
ax[0].set_title("train loss"); ax[0].legend()
if len(eval_rows):
    ax[1].plot(eval_rows.step, eval_rows.eval_macro_f1, "o-", label="macro-F1")
    ax[1].plot(eval_rows.step, eval_rows.eval_accuracy, "s--", label="accuracy"); ax[1].legend()
ax[1].set_title("val subset"); ax[1].set_ylim(0, 1)
if len(train_rows) and "sec_per_step" in train_rows:
    ax[2].plot(train_rows.step, train_rows.sec_per_step, label="s/step")
    ax2 = ax[2].twinx(); ax2.plot(train_rows.step, train_rows.get("device_mem_gb"), c="tab:red", label="GB"); ax2.set_ylabel("device GB")
ax[2].set_title("speed / memory")
for a in ax: a.set_xlabel("optimizer step")
plt.tight_layout(); plt.show()

In [ ]:
from manuai import report
metrics = RUN / "training_metrics.json"
display(Markdown(report.render(json.loads(metrics.read_text())))) if metrics.exists() else print("no training_metrics.json yet (run still going?)")